In [21]:
import pandas as pd
import yaml

In [22]:
def load_rules(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)["attacks"]
    
def convert_timestamps(df):
    # Zeek timestamps are Unix epoch (UTC)
    df["datetime"] = pd.to_datetime(df["ts"], unit="s", utc=True)
    # Convert to CICIDS2017 experiment timezone
    df["datetime"] = df["datetime"].dt.tz_convert("Canada/Atlantic")
    df["time"] = df["datetime"].dt.time
    return df

def label_flows_vectorized(df, rules):
    df["label"] = "BENIGN"

    for rule in rules:
        start = pd.to_datetime(rule["start"]).time()
        end = pd.to_datetime(rule["end"]).time()

        attacker_ips = rule["attacker_ips"]
        victim_ips = rule["victim_ips"]

        mask = (
            (df["time"] >= start) &
            (df["time"] <= end) &
            (
                (
                    df["id.orig_h"].isin(attacker_ips) &
                    df["id.resp_h"].isin(victim_ips)
                )
                |
                (
                    df["id.orig_h"].isin(victim_ips) &
                    df["id.resp_h"].isin(attacker_ips)
                )
            )
        )

        df.loc[mask, "label"] = rule["label"]

        print(rule["name"], "matches:", mask.sum())

    return df

In [23]:
df = pd.read_csv("zeek_logs/conn.tsv", delimiter="\t")
print("Loaded flows:", len(df))
df.head()

Loaded flows: 509362


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
0,1.499255e+09,CwTRCs3O6Ws5R6ccf4,192.168.10.14,49459,209.48.71.168,80,tcp,-,0.038308,0,...,T,F,0,FfA,2,80,1,40,-,6
1,1.499255e+09,CvFenhgh3byJg0l62,192.168.10.17,34492,192.168.10.3,53,udp,dns,0.000224,46,...,T,T,0,Dd,2,102,2,252,-,17
2,1.499255e+09,CEb1F3hxNxK8COYbf,192.168.10.17,15221,192.168.10.3,53,udp,dns,0.000190,46,...,T,T,0,Dd,2,102,2,252,-,17
3,1.499255e+09,ClicUT2W6Xj8VUlZN4,192.168.10.17,137,192.168.10.50,137,udp,dns,0.000003,62,...,T,T,0,D^d,1,90,1,90,-,17
4,1.499255e+09,CoHDza2uHEDMXVK7Wi,192.168.10.17,3186,192.168.10.3,53,udp,dns,0.000216,68,...,T,T,0,Dd,2,124,2,188,-,17


In [24]:
df = convert_timestamps(df)

print("Dataset datetime range:")
print(df["datetime"].min())
print(df["datetime"].max())

print("\nDataset time range:")
print(df["time"].min(), "->", df["time"].max())

Dataset datetime range:
2017-07-05 08:42:42.084372044-03:00
2017-07-05 17:10:14.557249069-03:00

Dataset time range:
08:42:42.084372 -> 17:10:14.557249


In [25]:
rules = load_rules("rules/wednesday.yaml")

In [26]:
df = label_flows_vectorized(df, rules)

DoS_Slowloris matches: 3864
DoS_Hulk matches: 154769
DoS_GoldenEye matches: 7739


In [27]:
df["label"].value_counts()

label
BENIGN            342990
DoS-HTTP_Flood    154769
DOS_GOLDENEYE       7739
DOS_SLOWLORIS       3864
Name: count, dtype: int64

In [ ]:
df.drop(columns=["datetime", "time"], inplace=True)

df.to_csv("wednesday_labeled.tsv", sep='\t', index=False, header=True)

print("Saved labeled dataset: wednesday_labeled.tsv")

Saved labeled dataset: wednesday_labeled.csv
